In [2]:
import torch
import torch.nn.functional as F
import requests

In [3]:
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
words = requests.get(url).text.splitlines()
print(words.__len__())

if device := torch.device("mps" if torch.backends.mps.is_available() else "cpu"):
    print(f"Using device: {device}")


print(words[:10])
len(words)
block_size = 7
batch_size = words.__len__()
epochs = 2000
lr = 0.1

32033
Using device: mps
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia', 'harper', 'evelyn']


In [4]:
chars = ['.'] + sorted(set(''.join(words)))
def stoi(s):
    return {c: i for i, c in enumerate(chars)}[s]
def itos(i):
    return {i: c for i, c in enumerate(chars)}[i]
print(itos(1), itos(2), itos(3), itos(0))
print(stoi('.'), stoi('a'), stoi('b'), stoi('c'))

a b c .
0 1 2 3


In [5]:
# encodes words indivdually
# def getXY(word):
#     X, Y = [], []
#     context = [0] * block_size
#     for ch in word + '.':
#         ix = stoi(ch)
#         X.append(context)
#         Y.append(ix)
#         context = context[1:] + [ix]
#         print(context)
#     return torch.tensor(X), torch.tensor(Y)

In [6]:
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi(ch)
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [7]:
X, Y = build_dataset(words)
Xtr, Ytr = X[:30000], Y[:30000]
Xdev, Ydev = X[30000:], Y[30000:]
print(X.shape, Y.shape)
print(Xtr.shape, Ytr.shape)
print(Xdev.shape, Ydev.shape)

torch.Size([228146, 7]) torch.Size([228146])
torch.Size([30000, 7]) torch.Size([30000])
torch.Size([198146, 7]) torch.Size([198146])


In [23]:
C = torch.randn((27, 2),requires_grad=True)
W1 = torch.randn((14, 100), requires_grad=True)
b1 = torch.randn(100, requires_grad=True)
W2 = torch.randn((100, 100), requires_grad=True)
b2 = torch.randn(100, requires_grad=True)
W3 = torch.randn((100, 27), requires_grad=True)
b3 = torch.randn(27, requires_grad=True)
optimizer = torch.optim.AdamW([C, W1, b1, W2, b2, W3, b3], lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.999)


In [14]:
for e in range(epochs):
    ix = torch.randint(0, Xtr.shape[0], (batch_size,))
    Xb, Yb = Xtr[ix], Ytr[ix]
    # forward
    emb = C[Xb]
    emb = emb.view(-1, 14)
    h = torch.tanh(emb @ W1 + b1)
    g = torch.tanh(h @ W2 + b2)
    logits = g @ W3 + b3


    loss = F.cross_entropy(logits, Yb)
    # backward
    loss.backward()
    # update
    optimizer.step()
    optimizer.zero_grad()
    scheduler.step()
    if e % 250 == 0:
        print(loss.item(), scheduler.get_last_lr() [0])

1.7514325380325317 0.01350647254721025
1.746176838874817 0.010517535744931123
1.732812523841858 0.008190040571973917
1.7277311086654663 0.006377612227551123
1.709761619567871 0.004966268160403829
1.715916395187378 0.0038672497732762327
1.7152575254440308 0.0030114404470033664
1.708203911781311 0.002345018837033896


In [19]:
loss = 0;
for i in range(Xdev.shape[0]):
    context = Xdev[i].tolist()
    target = Ydev[i].item()
    emb = C[torch.tensor([context])] # (1, 3, 2)
    emb = emb.view(1, -1) # (1, 10)
    h = torch.tanh(emb @ W1 + b1) # (1, 100)
    g = torch.tanh(h @ W2 + b2) # (1, 100)
    logits = g @ W3 + b3 # (1, 27)
    probs = F.softmax(logits, dim=1)
    loss += -probs[0, target].log().item()
print(loss / Xdev.shape[0])


2.807568394469526


In [21]:
cnt = 10
for i in range(cnt):
    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])] # (1, 3, 2)
        emb = emb.view(1, -1) # (1, 10)
        h = torch.tanh(emb @ W1 + b1) # (1, 100)
        g = torch.tanh(h @ W2 + b2) # (1, 100)
        logits = g @ W3 + b3 # (1, 27)
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break
    print(''.join(itos(i) for i in out))

aila.
emilei.
sirenne.
gilyanne.
omlou.
bicileo.
mailani.
moriana.
resnitievte.
prriste.
